# Quick sanity check for English responses

이 노트북은 비디오에서 한 프레임을 캡쳐한 뒤 InternVL 모델에 전달하여 영어 JSON 응답을 확인하기 위한 간단한 예제입니다.

In [ ]:
import os, sys, torch, cv2, numpy as np
from pathlib import Path
from PIL import Image
from IPython.display import display

# 프로젝트 루트를 PYTHONPATH에 추가해 평가 스크립트의 헬퍼를 재사용합니다.
sys.path.append(os.getcwd())
print(f"Current working directory: {os.getcwd()}")
print(f"CUDA devices detected: {torch.cuda.device_count()}")

: 

In [ ]:
VIDEO_PATH = Path('data/processed/gangnam/QA/qulity_test/clip_vio/gaepo1_shadow_clip.mp4')
FRAME_INDEX = 0  # 필요 시 다른 프레임으로 변경

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise FileNotFoundError(f'Cannot open video: {VIDEO_PATH}')

cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_INDEX)
ok, frame_bgr = cap.read()
cap.release()
if not ok or frame_bgr is None:
    raise RuntimeError('Failed to read a frame from the video')

frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
test_image = Image.fromarray(frame_rgb)
print(f'Captured frame shape: {frame_rgb.shape}')
display(test_image)

: 

In [ ]:
from src.evaluation.evaluate_qualitative_video_violence import (
    load_model_and_tokenizer,
    frames_to_pixel_values,
    PROMPT_TEMPLATE,
    parse_prediction,
)

checkpoint_path = 'ckpts/InternVL3-2B_gangnam_yeoksam_v2_gaepo4_1v2_samsung_rwf2000'
num_gpus = torch.cuda.device_count()
if num_gpus == 0:
    raise RuntimeError('CUDA 장치를 찾을 수 없습니다. GPU 환경에서 실행하세요.')

use_multi_gpu = num_gpus > 1
model, tokenizer, image_size = load_model_and_tokenizer(
    checkpoint_path,
    multi_gpu=use_multi_gpu,
)
device = next(model.parameters()).device
print(f'Loaded model on {device} (multi_gpu={use_multi_gpu}) | image_size={image_size}')

In [ ]:
frames_rgb = [np.array(test_image)]
pixel_values_tensor, num_patches_list = frames_to_pixel_values(frames_rgb, image_size)
device = next(model.parameters()).device
pixel_values_tensor = pixel_values_tensor.to(device).to(torch.bfloat16)

question = 'Frame1: <image>
' + PROMPT_TEMPLATE
response = model.chat(
    tokenizer,
    pixel_values=pixel_values_tensor,
    question=question,
    generation_config=dict(num_beams=1, do_sample=False, max_new_tokens=15, min_new_tokens=5),
    num_patches_list=num_patches_list,
    history=None,
    return_history=False,
)
print('Raw response:')
print(response)
print('
Parsed category:', parse_prediction(response))